In [5]:
# First, downgrade opencv to be compatible with numpy 1.x
!pip install 'opencv-python-headless<4.10' 'numpy>=1.23.5,<2.0'

# Then install cellpose without GUI (since GUI dependencies force numpy 2.x)
!pip install cellpose

# Verify installation
!pip list | grep -E "numpy|cellpose|opencv"

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
cellpose                     4.0.8
numpy                        1.26.4
opencv-python                4.8.0.74
opencv-python-headless       4.9.0.80


In [2]:
# !pip install 'zarr<3'
# !pip install timm


In [2]:
# ALWAYS RUN THIS FIRST!
import os
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Specify GPU 0 (out of 4 available GPUs)
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

NOTEBOOK_DIR = Path("/rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest")
os.chdir(NOTEBOOK_DIR)
sys.path.insert(0, str(NOTEBOOK_DIR))
print(f"✅ Working directory: {os.getcwd()}")
print(f"✅ Using GPU: {os.environ.get('CUDA_VISIBLE_DEVICES', 'Not set')}")

✅ Working directory: /rsrch9/home/plm/idso_fa1_pathology/codes/yshokrollahi/vitamin-p-latest
✅ Using GPU: 1


In [6]:
import os
import glob
import json
import re
import numpy as np
import cv2
from PIL import Image
from collections import defaultdict
from cellpose import models

from metrics import get_fast_pq, aggregated_jaccard_index

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
BASE_DIR        = "/rsrch9/home/plm/idso_fa1_pathology/TIER2/yasin-vitaminp/Pathology_val_dsp_bk"
CELLPOSE_MODEL  = 'nuclei'   # or 'cyto3' if you want to try, but nuclei is correct here
CELLPOSE_DIAM   = None       # None = auto-estimate per image
CELLPOSE_FLOW   = 0.4        # flow_threshold
CELLPOSE_CELLP  = 0.0        # cellprob_threshold
USE_GPU         = True

# ─────────────────────────────────────────────────────────────────
# MODEL  ← only this block changed
# ─────────────────────────────────────────────────────────────────
from cellpose import models

print("Loading CellPose model...")
cp_model = models.CellposeModel(gpu=USE_GPU)   # defaults to cpsam (CP4)
print("✅ CellPose ready\n")

# ─────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────
def canonical_type(class_name, object_type):
    text = (class_name + ' ' + object_type).lower()
    return 'Cell' if any(k in text for k in ['cytoplasm', 'cell']) else 'Nucleus'


def geojson_to_instance_map(geojson_path, image_shape, ann_type):
    """Rasterize GeoJSON annotations into an integer instance label map."""
    with open(geojson_path, 'r') as f:
        data = json.load(f)

    instance_map = np.zeros(image_shape[:2], dtype=np.int32)
    inst_id = 1

    for feature in data.get('features', []):
        props      = feature.get('properties', {})
        clf        = props.get('classification', {})
        class_name = clf.get('name', '') if isinstance(clf, dict) else str(clf or '')
        if canonical_type(class_name, props.get('objectType', '')) != ann_type:
            continue

        coords = feature['geometry']['coordinates']
        while (isinstance(coords, list) and coords
               and isinstance(coords[0], list)
               and not isinstance(coords[0][0], (int, float))):
            coords = coords[0]

        pts = np.array(coords, dtype=np.int32).reshape(-1, 1, 2)
        cv2.fillPoly(instance_map, [pts], inst_id)
        inst_id += 1

    return instance_map


def run_cellpose(he_img, model):
    """
    CP4 / Cellpose-SAM: no channels arg, returns (masks, flows, styles) — no diams.
    he_img: H x W x 3 uint8 RGB numpy array
    """
    masks, flows, styles = model.eval(
        he_img,
        diameter=CELLPOSE_DIAM,
        flow_threshold=CELLPOSE_FLOW,
        cellprob_threshold=CELLPOSE_CELLP,
        do_3D=False
    )
    return masks.astype(np.int32)

# ─────────────────────────────────────────────────────────────────
# METRICS
# ─────────────────────────────────────────────────────────────────
def binary_dice(pred_inst, gt_inst):
    pred = (pred_inst > 0).astype(np.uint8).flatten()
    gt   = (gt_inst   > 0).astype(np.uint8).flatten()
    intersection = (pred & gt).sum()
    denom        = pred.sum() + gt.sum()
    return float(2.0 * intersection / denom) if denom > 0 else 1.0


def detection_f1(pred_inst, gt_inst, iou_threshold=0.5):
    pred_ids = [p for p in np.unique(pred_inst) if p != 0]
    gt_ids   = [g for g in np.unique(gt_inst)   if g != 0]

    if len(gt_ids) == 0 and len(pred_ids) == 0:
        return 1.0, 1.0, 1.0
    if len(gt_ids) == 0:
        return 0.0, 0.0, 1.0
    if len(pred_ids) == 0:
        return 0.0, 1.0, 0.0

    tp = 0
    matched_pred = set()

    for g_id in gt_ids:
        gt_mask       = (gt_inst == g_id)
        overlap_preds = [p for p in np.unique(pred_inst[gt_mask])
                         if p != 0 and p not in matched_pred]
        best_iou, best_pid = 0.0, None

        for p_id in overlap_preds:
            pred_mask    = (pred_inst == p_id)
            intersection = np.logical_and(gt_mask, pred_mask).sum()
            union        = np.logical_or(gt_mask,  pred_mask).sum()
            iou = intersection / union if union > 0 else 0.0
            if iou > best_iou:
                best_iou, best_pid = iou, p_id

        if best_iou >= iou_threshold and best_pid is not None:
            tp += 1
            matched_pred.add(best_pid)

    fp        = len(pred_ids) - tp
    fn        = len(gt_ids)   - tp
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return f1, precision, recall


def compute_all_metrics(pred_inst, gt_inst):
    results = {}
    results['dice'] = binary_dice(pred_inst, gt_inst)

    if gt_inst.max() == 0:
        results.update({'aji': 0.0, 'pq': 0.0, 'dq': 0.0, 'sq': 0.0,
                        'f1': 0.0, 'precision': 0.0, 'recall': 0.0})
        return results

    results['aji'] = float(aggregated_jaccard_index(gt_inst, pred_inst))

    pq_result = get_fast_pq(gt_inst, pred_inst)
    if isinstance(pq_result, (list, tuple, np.ndarray)):
        pq_vals      = np.array(pq_result).flatten()
        results['pq'] = float(pq_vals[0])
        results['dq'] = float(pq_vals[1]) if len(pq_vals) > 1 else 0.0
        results['sq'] = float(pq_vals[2]) if len(pq_vals) > 2 else 0.0
    else:
        results['pq'] = float(pq_result)
        results['dq'] = 0.0
        results['sq'] = 0.0

    f1, prec, rec    = detection_f1(pred_inst, gt_inst, iou_threshold=0.5)
    results['f1']        = f1
    results['precision'] = prec
    results['recall']    = rec

    return results


# ─────────────────────────────────────────────────────────────────
# SAMPLE DISCOVERY
# ─────────────────────────────────────────────────────────────────
def get_all_sample_paths(base_dir):
    sample_paths = []
    for sample_dir in sorted(glob.glob(os.path.join(base_dir, "MOS*"))):
        if not os.path.isdir(sample_dir):
            continue
        sample_id = os.path.basename(sample_dir)
        all_files = glob.glob(os.path.join(sample_dir, "*"))
        img_files = [f for f in all_files
                     if any(f.lower().endswith(e) for e in ['.png','.jpg','.jpeg','.tif','.tiff'])]
        he_images = [f for f in img_files
                     if re.search(r'[_\-](he|hne)[_\-.]', os.path.basename(f), re.IGNORECASE)
                     and 'mif' not in f.lower()]
        geojsons  = sorted(f for f in all_files if f.endswith('.geojson'))

        if not he_images or not geojsons:
            print(f"[SKIP] {sample_id} — missing H&E or GeoJSON")
            continue

        sample_paths.append({
            'sample_id':    sample_id,
            'he_path':      he_images[0],
            'geojson_path': geojsons[-1],
        })
    return sample_paths


# ─────────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────────
all_samples = get_all_sample_paths(BASE_DIR)
print(f"Found {len(all_samples)} samples\n")

METRIC_KEYS = ['dice', 'aji', 'pq', 'dq', 'sq', 'f1', 'precision', 'recall']
col_w = 10

header_metrics = ''.join(f"{m:>{col_w}}" for m in METRIC_KEYS)
print(f"{'Sample':<12}  {'Type':<8}" + header_metrics)
print("-" * (22 + col_w * len(METRIC_KEYS)))

all_results = defaultdict(lambda: defaultdict(list))

for s in all_samples:
    sid = s['sample_id']
    try:
        he_img = np.array(Image.open(s['he_path']).convert('RGB'))

        # ── CellPose inference (nuclei only) ──
        nuclei_inst = run_cellpose(he_img, cp_model)

        # ── GT instance map (nuclei only) ──
        gt_nuclei_inst = geojson_to_instance_map(s['geojson_path'], he_img.shape, 'Nucleus')

        # ── Metrics ──
        nuc_metrics = compute_all_metrics(nuclei_inst, gt_nuclei_inst)

        row = ''.join(f"{nuc_metrics[m]:>{col_w}.4f}" for m in METRIC_KEYS)
        print(f"{sid:<12}  {'Nuclei':<8}" + row)

        for m in METRIC_KEYS:
            all_results['nuclei'][m].append(nuc_metrics[m])

    except Exception as e:
        print(f"{sid:<12}  ERROR: {e}")

# ─────────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────────
print("\n" + "=" * (22 + col_w * len(METRIC_KEYS)))
print("SUMMARY — CellPose Nuclei")
print("=" * (22 + col_w * len(METRIC_KEYS)))
print(f"{'Stat':<12}  {'Type':<8}" + header_metrics)
print("-" * (22 + col_w * len(METRIC_KEYS)))

for stat_name, fn in [('Mean', np.mean), ('Std', np.std), ('Min', np.min), ('Max', np.max)]:
    row = ''.join(f"{fn(all_results['nuclei'][m]):>{col_w}.4f}" for m in METRIC_KEYS)
    print(f"{stat_name:<12}  {'nuclei':<8}" + row)

print("\n✅ Done.")

Loading CellPose model...
✅ CellPose ready

Found 24 samples

Sample        Type          dice       aji        pq        dq        sq        f1 precision    recall
------------------------------------------------------------------------------------------------------
MOS001        Nuclei      0.8372    0.6795    0.6688    0.8822    0.7581    0.8858    0.8601    0.9130
MOS002        Nuclei      0.8351    0.6576    0.6127    0.8335    0.7351    0.8374    0.8173    0.8586
MOS003        Nuclei      0.8150    0.5741    0.5976    0.8023    0.7449    0.8023    0.8932    0.7282
MOS004        Nuclei      0.8697    0.7164    0.7029    0.9078    0.7742    0.9078    0.9491    0.8700
MOS008        Nuclei      0.8257    0.6991    0.7008    0.9138    0.7669    0.9138    0.9550    0.8760
MOS010        Nuclei      0.7634    0.5975    0.6448    0.8553    0.7538    0.8553    0.8471    0.8636
MOS014        Nuclei      0.8560    0.7204    0.6822    0.8740    0.7805    0.8780    0.9292    0.8321
MOS015     